In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from tqdm import tqdm
from urllib.parse import quote

import time
import re
import pandas as pd

# =========================
# 설정값(원하는 대로 변경)
# =========================
SOURCE_NAME = "네이버_블로그"   # ✅ 출처 컬럼에 들어갈 값(원하면 변경)
QUERY_RAW = '"육아" 힘들 | 불안 | 걱정 | 어렵 | 불편 -광고 -협찬'  # 사용자 검색어
DATE = "from20250802to20260301"  # nso 기간 (원하면 변경)
PAUSE_SEC = 1.1                  # 스크롤/로딩 대기
MAX_NO_CHANGE = 3                # 스크롤 높이 변화 없을 때 종료 기준

# =========================
# 공통 유틸
# =========================
def normalize_yyyy_mm_dd(date_raw: str) -> str:
    """
    어떤 형태든 (2026. 2. 7. / 2026-02-07 / 2026.02.07) -> YYYY-MM-DD 로 정규화
    실패하면 "" 반환
    """
    date_raw = (date_raw or "").strip()
    m = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})", date_raw)
    if not m:
        return ""
    y = int(m.group(1))
    mo = int(m.group(2))
    d = int(m.group(3))
    return f"{y:04d}-{mo:02d}-{d:02d}"


def scroll_to_bottom(driver, pause_sec=1.1, max_no_change=3):
    """
    검색 결과 페이지에서 끝까지 스크롤
    """
    no_change = 0
    last_h = driver.execute_script("return document.documentElement.scrollHeight")

    while True:
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(pause_sec)

        new_h = driver.execute_script("return document.documentElement.scrollHeight")
        if new_h == last_h:
            no_change += 1
            if no_change >= max_no_change:
                break
        else:
            no_change = 0
            last_h = new_h


def click_more_if_exists(driver):
    """
    '더보기' 류 버튼이 있으면 클릭 시도 (없으면 그냥 패스)
    - 네이버 DOM이 자주 바뀌므로 여러 후보를 시도
    """
    candidates = [
        (By.CSS_SELECTOR, "a.more_link"),     # 사용자가 쓰던 more_link
        (By.CSS_SELECTOR, "a.api_more"),      # 종종 쓰이는 더보기
        (By.LINK_TEXT, "더보기"),
    ]
    for by, sel in candidates:
        try:
            els = driver.find_elements(by, sel)
            if els:
                els[0].click()
                time.sleep(0.7)
                return True
        except:
            pass
    return False


def collect_result_links(driver):
    """
    검색 결과(블로그 탭)에서 '진짜 게시글 링크'만 href 수집
    - blog.naver.com/<blogId>/<숫자postNo> 형태만 통과
    - MyBlog.naver / section.blog.naver.com 같은 잡링크 제거
    """
    a_tags = driver.find_elements(
        By.CSS_SELECTOR,
        "a[href*='blog.naver.com'], a[href*='m.blog.naver.com']"
    )

    hrefs = []
    for a in a_tags:
        href = (a.get_attribute("href") or "").strip()
        if not href:
            continue

        # ✅ 게시글 링크만 필터 (postNo가 숫자인 형태)
        # 예: https://blog.naver.com/ojin_nio/223252158717
        if re.search(r"^https?://(m\.)?blog\.naver\.com/[^/]+/\d+", href):
            hrefs.append(href)

    # 중복 제거(순서 유지)
    seen = set()
    hrefs_u = []
    for h in hrefs:
        if h not in seen:
            seen.add(h)
            hrefs_u.append(h)

    return hrefs_u


# =========================
# 네이버 블로그 본문/제목/작성일 추출
# =========================
def extract_naver_title(driver) -> str:
    """
    제목: 요청하신 그림2 기반( span 태그 안 텍스트 )
    - 우선순위:
      1) og:title (가장 안정)
      2) span.se-fs.se-ff (그림2의 형태)
      3) h1/h2 fallback
    """
    # 1) og:title
    try:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[property='og:title']")
        if metas:
            t = (metas[0].get_attribute("content") or "").strip()
            if t:
                return t
    except:
        pass

    # 2) 그림2 기반: span.se-fs.se-ff (다만 본문에도 많아서 '제목 후보 영역' 먼저 탐색)
    # 네이버 블로그 제목은 종종 se-title-text, se-title 등 영역에 있음
    title_candidates = [
        "div.se-module.se-module-text.se-title-text span.se-fs.se-ff",
        "div.se-title-text span.se-fs.se-ff",
        "h1 span.se-fs.se-ff",
        "h2 span.se-fs.se-ff",
    ]
    for sel in title_candidates:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            for e in els:
                t = (e.text or "").strip()
                if t:
                    return t
        except:
            pass

    # 3) fallback
    for sel in ["h1", "h2", "title"]:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            t = (el.text or "").strip()
            if t:
                return t
        except:
            pass

    return ""


def extract_naver_content(driver) -> str:
    """
    본문: div.se-main-container 전체 텍스트를 우선 수집하되,
    동영상 위젯(pzp: 재생/좋아요/공유 등) 잡음 영역은 제거 후 추출
    """
    try:
        conts = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container")
        if not conts:
            return ""
        cont = conts[0]

        remove_selectors = [
            # 동영상(pzp) UI
            "div.pzp-pc__content-info",
            "[class*='pzp-pc__content-info']",
            "[class*='pzp-']",
        
            # 오픈그래프(링크 카드)
            "div.se-oglink-info-container",
            "[class*='se-oglink-info-container']",
        
            # ✅ 지도(본문에 붙는 위치/지도 위젯)
            "div.__se_map.se-dynamic-map",          # 핵심 (클래스 2개)
            "div.se-module.se-module-map-text",     # 지도 텍스트 모듈 (클래스 2개)
            "[class*='se-dynamic-map']",            # 변형 대비
            "[class*='se-module-map-text']",        # 변형 대비
        ]
        for sel in remove_selectors:
            try:
                bads = cont.find_elements(By.CSS_SELECTOR, sel)
                for b in bads:
                    driver.execute_script("arguments[0].remove();", b)
            except:
                pass

        # 본문 텍스트 추출
        t = (cont.text or "").strip()
        if t:
            return t

    except:
        pass

    # ---- fallback (se-main-container가 없거나 비어있는 케이스) ----
    # 텍스트 컴포넌트들을 모아서 합치기
    texts = []
    try:
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container div.se-component.se-text")
        for b in blocks:
            s = (b.text or "").strip()
            if s:
                texts.append(s)
    except:
        pass

    # 구형 에디터 fallback
    if not texts:
        for sel in ["#postViewArea", "div#viewTypeSelector + div", "div.post-view"]:
            try:
                els = driver.find_elements(By.CSS_SELECTOR, sel)
                if els:
                    s = (els[0].text or "").strip()
                    if s:
                        return s
            except:
                pass

    return "\n".join(texts).strip()


def extract_naver_date(driver) -> str:
    """
    작성일: YYYY-MM-DD로 정규화
    """
    date_raw = ""

    # 후보 셀렉터들 (케이스별)
    selectors = [
        "span.se_publishDate",   # 새 에디터
        "p.blog_date",           # 구형 스킨
        "span.date",
        "time",
    ]

    for sel in selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els:
                txt = (els[0].text or "").strip()
                if txt:
                    date_raw = txt
                    break
        except:
            pass

    return normalize_yyyy_mm_dd(date_raw)


# =========================
# 메인 크롤링
# =========================
driver = wb.Chrome()
wait = WebDriverWait(driver, 10)

key_word = quote(QUERY_RAW)
url = f"https://search.naver.com/search.naver?ssc=tab.blog.all&query={key_word}&nso=so%3Ar%2Cp%3A{DATE}"
driver.get(url)
time.sleep(1.2)

# (있으면) 더보기 한번 클릭 시도
click_more_if_exists(driver)

# 끝까지 스크롤
scroll_to_bottom(driver, pause_sec=PAUSE_SEC, max_no_change=MAX_NO_CHANGE)
print("검색 결과 스크롤 완료")

# 링크 수집
href_list = collect_result_links(driver)
print("수집된 링크 수:", len(href_list))

rows = []

# 게시글 수집(같은 driver를 재사용하는 게 더 안정적/가벼움)
for link in tqdm(href_list, desc="crawl posts"):
    try:
        driver.get(link)
        time.sleep(1.0)

        # 네이버 블로그는 mainFrame을 쓰는 경우가 많음
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame("mainFrame")
            time.sleep(0.3)
        except:
            # frame 없는 경우도 있음
            pass

        title = extract_naver_title(driver)
        content = extract_naver_content(driver)
        write_dt = extract_naver_date(driver)

        # --- 기존 코드 그대로 title/content/write_dt 추출한 다음 ---

        # ✅ (수정1) 내용이 없거나 작성일이 없으면 저장하지 않음
        #   - 필요하면 조건을 완화할 수도 있음(예: content만 필수)
        if not content.strip() or not write_dt.strip():
            continue
        
        rows.append({
            "출처": SOURCE_NAME,
            "제목": title,
            "내용": content,
            "작성일": write_dt,
            "링크": link
        })

    except Exception:
        # 실패한 글은 스킵
        continue

driver.quit()

df = pd.DataFrame(rows, columns=["출처", "제목", "내용", "작성일", "링크"])
df.to_csv(f"{SOURCE_NAME}_육아({len(df)}건).csv", index=False, encoding="utf-8-sig")
print("CSV 저장 완료:", f"{SOURCE_NAME}_육아({len(df)}건)_{DATE}.csv")

검색 결과 스크롤 완료
수집된 링크 수: 1012


crawl posts:  16%|██████████▌                                                       | 161/1012 [06:34<29:49,  2.10s/it]

In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from tqdm import tqdm
from urllib.parse import quote

import time
import re
import pandas as pd

# =========================
# 설정값(원하는 대로 변경)
# =========================
SOURCE_NAME = "네이버_블로그"   # ✅ 출처 컬럼에 들어갈 값(원하면 변경)
QUERY_RAW = '"육아" 힘들 | 불안 | 걱정 | 어렵 | 불편 -광고 -협찬'  # 사용자 검색어
DATE = "from20250302to20250801"  # nso 기간 (원하면 변경)
PAUSE_SEC = 1.1                  # 스크롤/로딩 대기
MAX_NO_CHANGE = 3                # 스크롤 높이 변화 없을 때 종료 기준

# =========================
# 공통 유틸
# =========================
def normalize_yyyy_mm_dd(date_raw: str) -> str:
    """
    어떤 형태든 (2026. 2. 7. / 2026-02-07 / 2026.02.07) -> YYYY-MM-DD 로 정규화
    실패하면 "" 반환
    """
    date_raw = (date_raw or "").strip()
    m = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})", date_raw)
    if not m:
        return ""
    y = int(m.group(1))
    mo = int(m.group(2))
    d = int(m.group(3))
    return f"{y:04d}-{mo:02d}-{d:02d}"


def scroll_to_bottom(driver, pause_sec=1.1, max_no_change=3):
    """
    검색 결과 페이지에서 끝까지 스크롤
    """
    no_change = 0
    last_h = driver.execute_script("return document.documentElement.scrollHeight")

    while True:
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(pause_sec)

        new_h = driver.execute_script("return document.documentElement.scrollHeight")
        if new_h == last_h:
            no_change += 1
            if no_change >= max_no_change:
                break
        else:
            no_change = 0
            last_h = new_h


def click_more_if_exists(driver):
    """
    '더보기' 류 버튼이 있으면 클릭 시도 (없으면 그냥 패스)
    - 네이버 DOM이 자주 바뀌므로 여러 후보를 시도
    """
    candidates = [
        (By.CSS_SELECTOR, "a.more_link"),     # 사용자가 쓰던 more_link
        (By.CSS_SELECTOR, "a.api_more"),      # 종종 쓰이는 더보기
        (By.LINK_TEXT, "더보기"),
    ]
    for by, sel in candidates:
        try:
            els = driver.find_elements(by, sel)
            if els:
                els[0].click()
                time.sleep(0.7)
                return True
        except:
            pass
    return False


def collect_result_links(driver):
    """
    검색 결과(블로그 탭)에서 '진짜 게시글 링크'만 href 수집
    - blog.naver.com/<blogId>/<숫자postNo> 형태만 통과
    - MyBlog.naver / section.blog.naver.com 같은 잡링크 제거
    """
    a_tags = driver.find_elements(
        By.CSS_SELECTOR,
        "a[href*='blog.naver.com'], a[href*='m.blog.naver.com']"
    )

    hrefs = []
    for a in a_tags:
        href = (a.get_attribute("href") or "").strip()
        if not href:
            continue

        # ✅ 게시글 링크만 필터 (postNo가 숫자인 형태)
        # 예: https://blog.naver.com/ojin_nio/223252158717
        if re.search(r"^https?://(m\.)?blog\.naver\.com/[^/]+/\d+", href):
            hrefs.append(href)

    # 중복 제거(순서 유지)
    seen = set()
    hrefs_u = []
    for h in hrefs:
        if h not in seen:
            seen.add(h)
            hrefs_u.append(h)

    return hrefs_u


# =========================
# 네이버 블로그 본문/제목/작성일 추출
# =========================
def extract_naver_title(driver) -> str:
    """
    제목: 요청하신 그림2 기반( span 태그 안 텍스트 )
    - 우선순위:
      1) og:title (가장 안정)
      2) span.se-fs.se-ff (그림2의 형태)
      3) h1/h2 fallback
    """
    # 1) og:title
    try:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[property='og:title']")
        if metas:
            t = (metas[0].get_attribute("content") or "").strip()
            if t:
                return t
    except:
        pass

    # 2) 그림2 기반: span.se-fs.se-ff (다만 본문에도 많아서 '제목 후보 영역' 먼저 탐색)
    # 네이버 블로그 제목은 종종 se-title-text, se-title 등 영역에 있음
    title_candidates = [
        "div.se-module.se-module-text.se-title-text span.se-fs.se-ff",
        "div.se-title-text span.se-fs.se-ff",
        "h1 span.se-fs.se-ff",
        "h2 span.se-fs.se-ff",
    ]
    for sel in title_candidates:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            for e in els:
                t = (e.text or "").strip()
                if t:
                    return t
        except:
            pass

    # 3) fallback
    for sel in ["h1", "h2", "title"]:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            t = (el.text or "").strip()
            if t:
                return t
        except:
            pass

    return ""


def extract_naver_content(driver) -> str:
    """
    본문: div.se-main-container 전체 텍스트를 우선 수집하되,
    동영상 위젯(pzp: 재생/좋아요/공유 등) 잡음 영역은 제거 후 추출
    """
    try:
        conts = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container")
        if not conts:
            return ""
        cont = conts[0]

        remove_selectors = [
            # 동영상(pzp) UI
            "div.pzp-pc__content-info",
            "[class*='pzp-pc__content-info']",
            "[class*='pzp-']",
        
            # 오픈그래프(링크 카드)
            "div.se-oglink-info-container",
            "[class*='se-oglink-info-container']",
        
            # ✅ 지도(본문에 붙는 위치/지도 위젯)
            "div.__se_map.se-dynamic-map",          # 핵심 (클래스 2개)
            "div.se-module.se-module-map-text",     # 지도 텍스트 모듈 (클래스 2개)
            "[class*='se-dynamic-map']",            # 변형 대비
            "[class*='se-module-map-text']",        # 변형 대비
        ]
        for sel in remove_selectors:
            try:
                bads = cont.find_elements(By.CSS_SELECTOR, sel)
                for b in bads:
                    driver.execute_script("arguments[0].remove();", b)
            except:
                pass

        # 본문 텍스트 추출
        t = (cont.text or "").strip()
        if t:
            return t

    except:
        pass

    # ---- fallback (se-main-container가 없거나 비어있는 케이스) ----
    # 텍스트 컴포넌트들을 모아서 합치기
    texts = []
    try:
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container div.se-component.se-text")
        for b in blocks:
            s = (b.text or "").strip()
            if s:
                texts.append(s)
    except:
        pass

    # 구형 에디터 fallback
    if not texts:
        for sel in ["#postViewArea", "div#viewTypeSelector + div", "div.post-view"]:
            try:
                els = driver.find_elements(By.CSS_SELECTOR, sel)
                if els:
                    s = (els[0].text or "").strip()
                    if s:
                        return s
            except:
                pass

    return "\n".join(texts).strip()


def extract_naver_date(driver) -> str:
    """
    작성일: YYYY-MM-DD로 정규화
    """
    date_raw = ""

    # 후보 셀렉터들 (케이스별)
    selectors = [
        "span.se_publishDate",   # 새 에디터
        "p.blog_date",           # 구형 스킨
        "span.date",
        "time",
    ]

    for sel in selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els:
                txt = (els[0].text or "").strip()
                if txt:
                    date_raw = txt
                    break
        except:
            pass

    return normalize_yyyy_mm_dd(date_raw)


# =========================
# 메인 크롤링
# =========================
driver = wb.Chrome()
wait = WebDriverWait(driver, 10)

key_word = quote(QUERY_RAW)
url = f"https://search.naver.com/search.naver?ssc=tab.blog.all&query={key_word}&nso=so%3Ar%2Cp%3A{DATE}"
driver.get(url)
time.sleep(1.2)

# (있으면) 더보기 한번 클릭 시도
click_more_if_exists(driver)

# 끝까지 스크롤
scroll_to_bottom(driver, pause_sec=PAUSE_SEC, max_no_change=MAX_NO_CHANGE)
print("검색 결과 스크롤 완료")

# 링크 수집
href_list = collect_result_links(driver)
print("수집된 링크 수:", len(href_list))

rows = []

# 게시글 수집(같은 driver를 재사용하는 게 더 안정적/가벼움)
for link in tqdm(href_list, desc="crawl posts"):
    try:
        driver.get(link)
        time.sleep(1.0)

        # 네이버 블로그는 mainFrame을 쓰는 경우가 많음
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame("mainFrame")
            time.sleep(0.3)
        except:
            # frame 없는 경우도 있음
            pass

        title = extract_naver_title(driver)
        content = extract_naver_content(driver)
        write_dt = extract_naver_date(driver)

        # --- 기존 코드 그대로 title/content/write_dt 추출한 다음 ---

        # ✅ (수정1) 내용이 없거나 작성일이 없으면 저장하지 않음
        #   - 필요하면 조건을 완화할 수도 있음(예: content만 필수)
        if not content.strip() or not write_dt.strip():
            continue
        
        rows.append({
            "출처": SOURCE_NAME,
            "제목": title,
            "내용": content,
            "작성일": write_dt,
            "링크": link
        })

    except Exception:
        # 실패한 글은 스킵
        continue

driver.quit()

df = pd.DataFrame(rows, columns=["출처", "제목", "내용", "작성일", "링크"])
df.to_csv(f"{SOURCE_NAME}_육아({len(df)}건).csv", index=False, encoding="utf-8-sig")
print("CSV 저장 완료:", f"{SOURCE_NAME}_육아({len(df)}건)_{DATE}.csv")

In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from tqdm import tqdm
from urllib.parse import quote

import time
import re
import pandas as pd

# =========================
# 설정값(원하는 대로 변경)
# =========================
SOURCE_NAME = "네이버_블로그"   # ✅ 출처 컬럼에 들어갈 값(원하면 변경)
QUERY_RAW = '"육아" 힘들 | 불안 | 걱정 | 어렵 | 불편 -광고 -협찬'  # 사용자 검색어
DATE = "from20240802to20250301"  # nso 기간 (원하면 변경)
PAUSE_SEC = 1.1                  # 스크롤/로딩 대기
MAX_NO_CHANGE = 3                # 스크롤 높이 변화 없을 때 종료 기준

# =========================
# 공통 유틸
# =========================
def normalize_yyyy_mm_dd(date_raw: str) -> str:
    """
    어떤 형태든 (2026. 2. 7. / 2026-02-07 / 2026.02.07) -> YYYY-MM-DD 로 정규화
    실패하면 "" 반환
    """
    date_raw = (date_raw or "").strip()
    m = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})", date_raw)
    if not m:
        return ""
    y = int(m.group(1))
    mo = int(m.group(2))
    d = int(m.group(3))
    return f"{y:04d}-{mo:02d}-{d:02d}"


def scroll_to_bottom(driver, pause_sec=1.1, max_no_change=3):
    """
    검색 결과 페이지에서 끝까지 스크롤
    """
    no_change = 0
    last_h = driver.execute_script("return document.documentElement.scrollHeight")

    while True:
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(pause_sec)

        new_h = driver.execute_script("return document.documentElement.scrollHeight")
        if new_h == last_h:
            no_change += 1
            if no_change >= max_no_change:
                break
        else:
            no_change = 0
            last_h = new_h


def click_more_if_exists(driver):
    """
    '더보기' 류 버튼이 있으면 클릭 시도 (없으면 그냥 패스)
    - 네이버 DOM이 자주 바뀌므로 여러 후보를 시도
    """
    candidates = [
        (By.CSS_SELECTOR, "a.more_link"),     # 사용자가 쓰던 more_link
        (By.CSS_SELECTOR, "a.api_more"),      # 종종 쓰이는 더보기
        (By.LINK_TEXT, "더보기"),
    ]
    for by, sel in candidates:
        try:
            els = driver.find_elements(by, sel)
            if els:
                els[0].click()
                time.sleep(0.7)
                return True
        except:
            pass
    return False


def collect_result_links(driver):
    """
    검색 결과(블로그 탭)에서 '진짜 게시글 링크'만 href 수집
    - blog.naver.com/<blogId>/<숫자postNo> 형태만 통과
    - MyBlog.naver / section.blog.naver.com 같은 잡링크 제거
    """
    a_tags = driver.find_elements(
        By.CSS_SELECTOR,
        "a[href*='blog.naver.com'], a[href*='m.blog.naver.com']"
    )

    hrefs = []
    for a in a_tags:
        href = (a.get_attribute("href") or "").strip()
        if not href:
            continue

        # ✅ 게시글 링크만 필터 (postNo가 숫자인 형태)
        # 예: https://blog.naver.com/ojin_nio/223252158717
        if re.search(r"^https?://(m\.)?blog\.naver\.com/[^/]+/\d+", href):
            hrefs.append(href)

    # 중복 제거(순서 유지)
    seen = set()
    hrefs_u = []
    for h in hrefs:
        if h not in seen:
            seen.add(h)
            hrefs_u.append(h)

    return hrefs_u


# =========================
# 네이버 블로그 본문/제목/작성일 추출
# =========================
def extract_naver_title(driver) -> str:
    """
    제목: 요청하신 그림2 기반( span 태그 안 텍스트 )
    - 우선순위:
      1) og:title (가장 안정)
      2) span.se-fs.se-ff (그림2의 형태)
      3) h1/h2 fallback
    """
    # 1) og:title
    try:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[property='og:title']")
        if metas:
            t = (metas[0].get_attribute("content") or "").strip()
            if t:
                return t
    except:
        pass

    # 2) 그림2 기반: span.se-fs.se-ff (다만 본문에도 많아서 '제목 후보 영역' 먼저 탐색)
    # 네이버 블로그 제목은 종종 se-title-text, se-title 등 영역에 있음
    title_candidates = [
        "div.se-module.se-module-text.se-title-text span.se-fs.se-ff",
        "div.se-title-text span.se-fs.se-ff",
        "h1 span.se-fs.se-ff",
        "h2 span.se-fs.se-ff",
    ]
    for sel in title_candidates:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            for e in els:
                t = (e.text or "").strip()
                if t:
                    return t
        except:
            pass

    # 3) fallback
    for sel in ["h1", "h2", "title"]:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            t = (el.text or "").strip()
            if t:
                return t
        except:
            pass

    return ""


def extract_naver_content(driver) -> str:
    """
    본문: div.se-main-container 전체 텍스트를 우선 수집하되,
    동영상 위젯(pzp: 재생/좋아요/공유 등) 잡음 영역은 제거 후 추출
    """
    try:
        conts = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container")
        if not conts:
            return ""
        cont = conts[0]

        remove_selectors = [
            # 동영상(pzp) UI
            "div.pzp-pc__content-info",
            "[class*='pzp-pc__content-info']",
            "[class*='pzp-']",
        
            # 오픈그래프(링크 카드)
            "div.se-oglink-info-container",
            "[class*='se-oglink-info-container']",
        
            # ✅ 지도(본문에 붙는 위치/지도 위젯)
            "div.__se_map.se-dynamic-map",          # 핵심 (클래스 2개)
            "div.se-module.se-module-map-text",     # 지도 텍스트 모듈 (클래스 2개)
            "[class*='se-dynamic-map']",            # 변형 대비
            "[class*='se-module-map-text']",        # 변형 대비
        ]
        for sel in remove_selectors:
            try:
                bads = cont.find_elements(By.CSS_SELECTOR, sel)
                for b in bads:
                    driver.execute_script("arguments[0].remove();", b)
            except:
                pass

        # 본문 텍스트 추출
        t = (cont.text or "").strip()
        if t:
            return t

    except:
        pass

    # ---- fallback (se-main-container가 없거나 비어있는 케이스) ----
    # 텍스트 컴포넌트들을 모아서 합치기
    texts = []
    try:
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container div.se-component.se-text")
        for b in blocks:
            s = (b.text or "").strip()
            if s:
                texts.append(s)
    except:
        pass

    # 구형 에디터 fallback
    if not texts:
        for sel in ["#postViewArea", "div#viewTypeSelector + div", "div.post-view"]:
            try:
                els = driver.find_elements(By.CSS_SELECTOR, sel)
                if els:
                    s = (els[0].text or "").strip()
                    if s:
                        return s
            except:
                pass

    return "\n".join(texts).strip()


def extract_naver_date(driver) -> str:
    """
    작성일: YYYY-MM-DD로 정규화
    """
    date_raw = ""

    # 후보 셀렉터들 (케이스별)
    selectors = [
        "span.se_publishDate",   # 새 에디터
        "p.blog_date",           # 구형 스킨
        "span.date",
        "time",
    ]

    for sel in selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els:
                txt = (els[0].text or "").strip()
                if txt:
                    date_raw = txt
                    break
        except:
            pass

    return normalize_yyyy_mm_dd(date_raw)


# =========================
# 메인 크롤링
# =========================
driver = wb.Chrome()
wait = WebDriverWait(driver, 10)

key_word = quote(QUERY_RAW)
url = f"https://search.naver.com/search.naver?ssc=tab.blog.all&query={key_word}&nso=so%3Ar%2Cp%3A{DATE}"
driver.get(url)
time.sleep(1.2)

# (있으면) 더보기 한번 클릭 시도
click_more_if_exists(driver)

# 끝까지 스크롤
scroll_to_bottom(driver, pause_sec=PAUSE_SEC, max_no_change=MAX_NO_CHANGE)
print("검색 결과 스크롤 완료")

# 링크 수집
href_list = collect_result_links(driver)
print("수집된 링크 수:", len(href_list))

rows = []

# 게시글 수집(같은 driver를 재사용하는 게 더 안정적/가벼움)
for link in tqdm(href_list, desc="crawl posts"):
    try:
        driver.get(link)
        time.sleep(1.0)

        # 네이버 블로그는 mainFrame을 쓰는 경우가 많음
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame("mainFrame")
            time.sleep(0.3)
        except:
            # frame 없는 경우도 있음
            pass

        title = extract_naver_title(driver)
        content = extract_naver_content(driver)
        write_dt = extract_naver_date(driver)

        # --- 기존 코드 그대로 title/content/write_dt 추출한 다음 ---

        # ✅ (수정1) 내용이 없거나 작성일이 없으면 저장하지 않음
        #   - 필요하면 조건을 완화할 수도 있음(예: content만 필수)
        if not content.strip() or not write_dt.strip():
            continue
        
        rows.append({
            "출처": SOURCE_NAME,
            "제목": title,
            "내용": content,
            "작성일": write_dt,
            "링크": link
        })

    except Exception:
        # 실패한 글은 스킵
        continue

driver.quit()

df = pd.DataFrame(rows, columns=["출처", "제목", "내용", "작성일", "링크"])
df.to_csv(f"{SOURCE_NAME}_육아({len(df)}건).csv", index=False, encoding="utf-8-sig")
print("CSV 저장 완료:", f"{SOURCE_NAME}_육아({len(df)}건)_{DATE}.csv")

In [ ]:
from selenium import webdriver as wb
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from tqdm import tqdm
from urllib.parse import quote

import time
import re
import pandas as pd

# =========================
# 설정값(원하는 대로 변경)
# =========================
SOURCE_NAME = "네이버_블로그"   # ✅ 출처 컬럼에 들어갈 값(원하면 변경)
QUERY_RAW = '"육아" 힘들 | 불안 | 걱정 | 어렵 | 불편 -광고 -협찬'  # 사용자 검색어
DATE = "from20240302to20240801"  # nso 기간 (원하면 변경)
PAUSE_SEC = 1.1                  # 스크롤/로딩 대기
MAX_NO_CHANGE = 3                # 스크롤 높이 변화 없을 때 종료 기준

# =========================
# 공통 유틸
# =========================
def normalize_yyyy_mm_dd(date_raw: str) -> str:
    """
    어떤 형태든 (2026. 2. 7. / 2026-02-07 / 2026.02.07) -> YYYY-MM-DD 로 정규화
    실패하면 "" 반환
    """
    date_raw = (date_raw or "").strip()
    m = re.search(r"(\d{4})\D+(\d{1,2})\D+(\d{1,2})", date_raw)
    if not m:
        return ""
    y = int(m.group(1))
    mo = int(m.group(2))
    d = int(m.group(3))
    return f"{y:04d}-{mo:02d}-{d:02d}"


def scroll_to_bottom(driver, pause_sec=1.1, max_no_change=3):
    """
    검색 결과 페이지에서 끝까지 스크롤
    """
    no_change = 0
    last_h = driver.execute_script("return document.documentElement.scrollHeight")

    while True:
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(pause_sec)

        new_h = driver.execute_script("return document.documentElement.scrollHeight")
        if new_h == last_h:
            no_change += 1
            if no_change >= max_no_change:
                break
        else:
            no_change = 0
            last_h = new_h


def click_more_if_exists(driver):
    """
    '더보기' 류 버튼이 있으면 클릭 시도 (없으면 그냥 패스)
    - 네이버 DOM이 자주 바뀌므로 여러 후보를 시도
    """
    candidates = [
        (By.CSS_SELECTOR, "a.more_link"),     # 사용자가 쓰던 more_link
        (By.CSS_SELECTOR, "a.api_more"),      # 종종 쓰이는 더보기
        (By.LINK_TEXT, "더보기"),
    ]
    for by, sel in candidates:
        try:
            els = driver.find_elements(by, sel)
            if els:
                els[0].click()
                time.sleep(0.7)
                return True
        except:
            pass
    return False


def collect_result_links(driver):
    """
    검색 결과(블로그 탭)에서 '진짜 게시글 링크'만 href 수집
    - blog.naver.com/<blogId>/<숫자postNo> 형태만 통과
    - MyBlog.naver / section.blog.naver.com 같은 잡링크 제거
    """
    a_tags = driver.find_elements(
        By.CSS_SELECTOR,
        "a[href*='blog.naver.com'], a[href*='m.blog.naver.com']"
    )

    hrefs = []
    for a in a_tags:
        href = (a.get_attribute("href") or "").strip()
        if not href:
            continue

        # ✅ 게시글 링크만 필터 (postNo가 숫자인 형태)
        # 예: https://blog.naver.com/ojin_nio/223252158717
        if re.search(r"^https?://(m\.)?blog\.naver\.com/[^/]+/\d+", href):
            hrefs.append(href)

    # 중복 제거(순서 유지)
    seen = set()
    hrefs_u = []
    for h in hrefs:
        if h not in seen:
            seen.add(h)
            hrefs_u.append(h)

    return hrefs_u


# =========================
# 네이버 블로그 본문/제목/작성일 추출
# =========================
def extract_naver_title(driver) -> str:
    """
    제목: 요청하신 그림2 기반( span 태그 안 텍스트 )
    - 우선순위:
      1) og:title (가장 안정)
      2) span.se-fs.se-ff (그림2의 형태)
      3) h1/h2 fallback
    """
    # 1) og:title
    try:
        metas = driver.find_elements(By.CSS_SELECTOR, "meta[property='og:title']")
        if metas:
            t = (metas[0].get_attribute("content") or "").strip()
            if t:
                return t
    except:
        pass

    # 2) 그림2 기반: span.se-fs.se-ff (다만 본문에도 많아서 '제목 후보 영역' 먼저 탐색)
    # 네이버 블로그 제목은 종종 se-title-text, se-title 등 영역에 있음
    title_candidates = [
        "div.se-module.se-module-text.se-title-text span.se-fs.se-ff",
        "div.se-title-text span.se-fs.se-ff",
        "h1 span.se-fs.se-ff",
        "h2 span.se-fs.se-ff",
    ]
    for sel in title_candidates:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            for e in els:
                t = (e.text or "").strip()
                if t:
                    return t
        except:
            pass

    # 3) fallback
    for sel in ["h1", "h2", "title"]:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            t = (el.text or "").strip()
            if t:
                return t
        except:
            pass

    return ""


def extract_naver_content(driver) -> str:
    """
    본문: div.se-main-container 전체 텍스트를 우선 수집하되,
    동영상 위젯(pzp: 재생/좋아요/공유 등) 잡음 영역은 제거 후 추출
    """
    try:
        conts = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container")
        if not conts:
            return ""
        cont = conts[0]

        remove_selectors = [
            # 동영상(pzp) UI
            "div.pzp-pc__content-info",
            "[class*='pzp-pc__content-info']",
            "[class*='pzp-']",
        
            # 오픈그래프(링크 카드)
            "div.se-oglink-info-container",
            "[class*='se-oglink-info-container']",
        
            # ✅ 지도(본문에 붙는 위치/지도 위젯)
            "div.__se_map.se-dynamic-map",          # 핵심 (클래스 2개)
            "div.se-module.se-module-map-text",     # 지도 텍스트 모듈 (클래스 2개)
            "[class*='se-dynamic-map']",            # 변형 대비
            "[class*='se-module-map-text']",        # 변형 대비
        ]
        for sel in remove_selectors:
            try:
                bads = cont.find_elements(By.CSS_SELECTOR, sel)
                for b in bads:
                    driver.execute_script("arguments[0].remove();", b)
            except:
                pass

        # 본문 텍스트 추출
        t = (cont.text or "").strip()
        if t:
            return t

    except:
        pass

    # ---- fallback (se-main-container가 없거나 비어있는 케이스) ----
    # 텍스트 컴포넌트들을 모아서 합치기
    texts = []
    try:
        blocks = driver.find_elements(By.CSS_SELECTOR, "div.se-main-container div.se-component.se-text")
        for b in blocks:
            s = (b.text or "").strip()
            if s:
                texts.append(s)
    except:
        pass

    # 구형 에디터 fallback
    if not texts:
        for sel in ["#postViewArea", "div#viewTypeSelector + div", "div.post-view"]:
            try:
                els = driver.find_elements(By.CSS_SELECTOR, sel)
                if els:
                    s = (els[0].text or "").strip()
                    if s:
                        return s
            except:
                pass

    return "\n".join(texts).strip()


def extract_naver_date(driver) -> str:
    """
    작성일: YYYY-MM-DD로 정규화
    """
    date_raw = ""

    # 후보 셀렉터들 (케이스별)
    selectors = [
        "span.se_publishDate",   # 새 에디터
        "p.blog_date",           # 구형 스킨
        "span.date",
        "time",
    ]

    for sel in selectors:
        try:
            els = driver.find_elements(By.CSS_SELECTOR, sel)
            if els:
                txt = (els[0].text or "").strip()
                if txt:
                    date_raw = txt
                    break
        except:
            pass

    return normalize_yyyy_mm_dd(date_raw)


# =========================
# 메인 크롤링
# =========================
driver = wb.Chrome()
wait = WebDriverWait(driver, 10)

key_word = quote(QUERY_RAW)
url = f"https://search.naver.com/search.naver?ssc=tab.blog.all&query={key_word}&nso=so%3Ar%2Cp%3A{DATE}"
driver.get(url)
time.sleep(1.2)

# (있으면) 더보기 한번 클릭 시도
click_more_if_exists(driver)

# 끝까지 스크롤
scroll_to_bottom(driver, pause_sec=PAUSE_SEC, max_no_change=MAX_NO_CHANGE)
print("검색 결과 스크롤 완료")

# 링크 수집
href_list = collect_result_links(driver)
print("수집된 링크 수:", len(href_list))

rows = []

# 게시글 수집(같은 driver를 재사용하는 게 더 안정적/가벼움)
for link in tqdm(href_list, desc="crawl posts"):
    try:
        driver.get(link)
        time.sleep(1.0)

        # 네이버 블로그는 mainFrame을 쓰는 경우가 많음
        try:
            driver.switch_to.default_content()
            driver.switch_to.frame("mainFrame")
            time.sleep(0.3)
        except:
            # frame 없는 경우도 있음
            pass

        title = extract_naver_title(driver)
        content = extract_naver_content(driver)
        write_dt = extract_naver_date(driver)

        # --- 기존 코드 그대로 title/content/write_dt 추출한 다음 ---

        # ✅ (수정1) 내용이 없거나 작성일이 없으면 저장하지 않음
        #   - 필요하면 조건을 완화할 수도 있음(예: content만 필수)
        if not content.strip() or not write_dt.strip():
            continue
        
        rows.append({
            "출처": SOURCE_NAME,
            "제목": title,
            "내용": content,
            "작성일": write_dt,
            "링크": link
        })

    except Exception:
        # 실패한 글은 스킵
        continue

driver.quit()

df = pd.DataFrame(rows, columns=["출처", "제목", "내용", "작성일", "링크"])
df.to_csv(f"{SOURCE_NAME}_육아({len(df)}건).csv", index=False, encoding="utf-8-sig")
print("CSV 저장 완료:", f"{SOURCE_NAME}_육아({len(df)}건)_{DATE}.csv")